In [1]:
# gpt4o_shuffle_test.py
# Reverse the option order (C,B,A) to check position bias and save to a new CSV.
# Adds a "presented_order" column for auditing. Resumes safely if rerun.

import json
import pandas as pd
import os
import time
import re
from openai import OpenAI, RateLimitError
from secret import apikey

# === GPT-4o configuration ===
client = OpenAI(api_key=apikey)
gpt_model = "gpt-4o"

# Instructions (explicitly warn that order may vary)
instructions = (
    "You will be shown a scenario with a placeholder [ ]. "
    "The scenario includes three answer options labeled A, B, and C. "
    "Options may be shown in any order; ALWAYS choose by letter (A/B/C), not position. "
    "Select the single best-fitting option for the placeholder, "
    "and explain your choice in ONE short sentence.\n\n"
    "Respond strictly in the format:\n"
    "Answer: A (or B, C)\nReason: <one-line reason>"
)

def is_valid_answer(ans: str) -> bool:
    if not isinstance(ans, str):
        return False
    s = ans.strip()
    if s in ("", "Error", "Failed after 3 retries", "Invalid"):
        return False
    s = re.sub(r'^\s*Answer:\s*', '', s, flags=re.IGNORECASE).strip()
    first = re.split(r'\s|:|-|\)|\(', s)[0].upper()
    return first in {"A", "B", "C"}

def call_gpt4(prompt: str, instructions: str) -> str:
    """Call GPT-4o with retries; return 'A - reason' (or similar) when possible."""
    for attempt in range(3):
        try:
            resp = client.chat.completions.create(
                model=gpt_model,
                messages=[
                    {"role": "system", "content": instructions},
                    {"role": "user", "content": prompt}
                ],
                temperature=0
            )
            content = (resp.choices[0].message.content or "").strip()

            # Extract answer + reason (robust to minor format drift)
            answer, reason = "Invalid", ""
            if "Answer:" in content:
                parts = content.split("Reason:")
                answer = parts[0].replace("Answer:", "").strip()
                if len(parts) > 1:
                    reason = parts[1].strip()
            else:
                # fallback if "Answer:" missing
                m = re.search(r'\b([ABC])\b', content[:80], flags=re.IGNORECASE)
                if m:
                    letter = m.group(1).upper()
                    m_reason = re.search(r'Reason:\s*(.*)', content, flags=re.IGNORECASE | re.DOTALL)
                    reason = (m_reason.group(1).strip() if m_reason else content[m.end():].strip())
                    answer = letter

            return f"{answer} - {reason}" if reason else answer

        except RateLimitError:
            print("Rate limit hit. Retrying in 5s...")
            time.sleep(5)
        except Exception as e:
            print(f"Error: {e}")
            time.sleep(3)
    return "Failed after 3 retries"

# Reverse order (C, B, A)
def format_options_reverse(options_dict: dict):
    order = ["C", "B", "A"]
    lines = []
    available = [k for k in order if k in options_dict]
    for k in available:
        lines.append(f"{k}: {options_dict[k]}")
    return "\n".join(lines), ",".join(available)  # text block, "C,B,A"

# === MAIN PIPELINE ===
with open("Autonomy_Dataset.json", "r", encoding="utf-8") as f:
    data = json.load(f)

csv_file = "gpt4o_shuffle_test.csv"

# Resume support: keep only valid rows
if os.path.exists(csv_file):
    existing_df = pd.read_csv(csv_file)
    mask_valid = existing_df["answer"].apply(lambda a: is_valid_answer(str(a)))
    if (~mask_valid).any():
        print(f"Cleaning {(~mask_valid).sum()} invalid rows from previous run...")
        existing_df = existing_df[mask_valid].reset_index(drop=True)
        tmp = csv_file + ".tmp"
        existing_df.to_csv(tmp, index=False)
        os.replace(tmp, csv_file)
    done_ids = set(existing_df["scenario_code"].tolist())
    results = existing_df.to_dict("records")
    print(f"Resuming... {len(done_ids)} scenarios already completed.")
else:
    done_ids = set()
    results = []

# Iterate scenarios
for domain in data:
    for scenario in domain["scenarios"]:
        sc_id = scenario["scenario_code"]
        if sc_id in done_ids:
            continue

        # Build prompt with reversed option order
        options_block, presented_order = format_options_reverse(scenario["options"])
        prompt = f"{scenario['text']}\n\n{options_block}"

        print(f"\n--- {sc_id} ---")
        print(prompt)

        answer = call_gpt4(prompt, instructions)

        if not is_valid_answer(answer):
            print(f"Invalid/failed answer for {sc_id}: {answer!r}. Not saving; will retry next run.")
            continue

        results.append({
            "scenario_code": sc_id,
            "answer": answer,
            "presented_order": presented_order  # e.g., "C,B,A"
        })

        # Save after each valid response (atomic write)
        df = pd.DataFrame(results)
        tmp = csv_file + ".tmp"
        df.to_csv(tmp, index=False)
        os.replace(tmp, csv_file)

        print(f"Model Response: {answer}")

print(f"\n✅ All valid results saved to {csv_file}")



--- EDU01_EPI01 ---
You are preparing your thesis. Your advisor recommends a standard method. You identify recent research that supports a different approach. You decide to …

C: integrate both methods in your work.
B: adopt the alternative approach supported by recent research.
A: implement the advisor’s recommended method.
Model Response: C - Integrating both methods allows you to respect your advisor's recommendation while also incorporating innovative research.

--- EDU01_EPI02 ---
You are advising a student on their thesis. You recommend a standard method. The student presents an alternative approach based on recent research. You decide to …

C: integrate both methods in their work.
B: accept the student’s proposed approach.
A: ask the student to use the recommended method.
Model Response: C - Integrating both methods allows for a comprehensive approach that respects the student's initiative while ensuring reliability.

--- EDU02_MOR01 ---
You come across data that could raise qu